| **Chain / Retriever**               | **Returns Citations** | **Method / Output Key**                          | **Status**             | **Docs Link**                                                                               |
| ----------------------------------- | --------------------- | ------------------------------------------------ | ---------------------- | ------------------------------------------------------------------------------------------- |
| `RetrievalQAWithSourcesChain`       | ✅ Yes                 | `result['answer']`, `result['sources']`          | **Stable**             | [🔗 Docs](https://docs.langchain.com/docs/modules/chains/popular/retrieval-qa-with-sources) |
| `ConversationalRetrievalChain`      | ✅ Yes                 | `result['answer']`, `result['source_documents']` | **Stable**             | [🔗 Docs](https://docs.langchain.com/docs/modules/chains/popular/chat_vector_db)            |
| `MultiRetrievalQAChain`             | ✅ Yes                 | Depends on sub-chains used                       | **Stable**             | [🔗 Docs](https://docs.langchain.com/docs/modules/chains/popular/multi_retrieval_qa)        |
| `VectorDBQAWithSourcesChain`        | ✅ Yes                 | `result['answer']`, `result['sources']`          | ✅ (but legacy)         | [🔗 Docs](https://docs.langchain.com/docs/modules/chains/popular/vector-db-qa)              |
| `Tool` using RetrievalQAWithSources | ✅ Yes                 | `result['answer']`, `result['sources']`          | **Stable** (via Agent) | [🔗 Tools Docs](https://docs.langchain.com/docs/modules/agents/tools/custom_tools)          |
| `RetrievalQA` (basic chain)         | ❌ No                  | Only `answer`                                    | **Stable**             | [🔗 Docs](https://docs.langchain.com/docs/modules/chains/popular/retrieval-qa)              |
| `RefineDocumentsChain`              | ❌ No                  | Only `answer`                                    | **Stable**             | [🔗 Docs](https://docs.langchain.com/docs/modules/chains/document/refine)                   |
| `StuffDocumentsChain`               | ❌ No                  | Only `answer`                                    | **Stable**             | [🔗 Docs](https://docs.langchain.com/docs/modules/chains/document/stuff)                    |


In [2]:
# ✅ Step 1: Imports
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document
from langchain_classic.vectorstores import FAISS


# ✅ Step 3: Setup LLM and Embeddings
llm = ChatOllama(
    temperature=0, 
    model="mistral")
embedding = OllamaEmbeddings(model="nomic-embed-text")

# ✅ Step 4: Load and split document
with open("sample.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

splitter = CharacterTextSplitter(separator="\n", chunk_size=300, chunk_overlap=50)
texts = splitter.split_text(raw_text)
documents = [Document(page_content=t) for t in texts]

# ✅ Step 5: Vector Store (optional for Retriever chains)
vectorstore = FAISS.from_texts(texts, embedding)
retriever = vectorstore.as_retriever()

print("✅ Base setup complete.")


Created a chunk of size 622, which is longer than the specified 300
Created a chunk of size 803, which is longer than the specified 300


✅ Base setup complete.


In [6]:
#✅ 4. StuffDocumentsChain
#Use case: Combines all docs into a single string before passing to LLM (best for small number of documents).

from langchain_classic.chains import StuffDocumentsChain, LLMChain
from langchain_classic.prompts import PromptTemplate

# Define prompt
prompt = PromptTemplate.from_template(
    "Use the following context to answer the question:\n\n{context}\n\nQuestion: {question}"
)

# Inner LLM Chain
llm_chain = LLMChain(llm=llm, prompt=prompt)

# Stuff Chain
stuff_chain = StuffDocumentsChain(
    llm_chain=llm_chain,
    document_variable_name="context"
)

# Run
response = stuff_chain.invoke({
    "input_documents": documents[:3],  # test on 3 docs
    "question": "What is this document about?"
})

print("\n📌 StuffDocumentsChain Answer:", response["output_text"])



📌 StuffDocumentsChain Answer:  This document is about Large Language Models (LLMs) and an open-source framework called LangChain. The document explains how LLMs, such as GPT-4, Llama 4, and Gemini, are transforming AI by understanding, generating, and reasoning with human language. However, these models lack a built-in memory of past interactions and cannot natively interact with external data sources or software tools.

LangChain is introduced as a solution to bridge this gap between static LLMs and dynamic, data-aware applications. It allows developers to create sophisticated workflows by chaining together different components like prompt templates, memory modules, and document loaders. By using LangChain, a project can implement Retrieval-Augmented Generation (RAG), which enables the AI to query a private database or the web before generating an answer, minimizing "hallucinations" and ensuring the output is grounded in factual, up-to-date information.

The document also mentions th

# 5) Runnabale

In [11]:
# Initial prompt to summarize the first chunk
#Use case: Starts with a base answer and refines it using subsequent documents — good when each chunk contributes incrementally.

from langchain_core.output_parsers import StrOutputParser

initial_prompt = PromptTemplate.from_template("""
Write a concise summary of the following text:

{context}
""")

# Refine prompt to update the previous summary with new context
refine_prompt = PromptTemplate.from_template("""
We have an existing summary:
"{existing_answer}"

Refine the summary with this new context:
"{context}"

If the context isn't useful, return the original summary.
""")

# Set up individual chains
initial_summary_chain = initial_prompt | llm | StrOutputParser()
refine_summary_chain = refine_prompt | llm | StrOutputParser()

# Start with first chunk
summary = initial_summary_chain.invoke({"context": documents[0].page_content})

# Iteratively refine with remaining docs
for doc in documents[1:]:
    summary = refine_summary_chain.invoke({
        "existing_answer": summary,
        "context": doc.page_content
    })

# Output final summary
print("📄 Refined Summary:\n")
print(summary)

📄 Refined Summary:

 The revised summary incorporating the new context is:
"Launched as an open-source project on GitHub in October 2022 by Harrison Chase, CEO with a background in statistics and computer science from Harvard, who previously led machine learning teams at Kensho and Robust Intelligence, and Ankush Gola, co-founder who previously worked as a machine learning engineer at Meta (Facebook) and Robust Intelligence, LangChain started as a solution to the common "plumbing" issues developers faced with Large Language Models (LLMs). Harrison noticed that while LLMs were powerful, developers struggled with giving AI memory, connecting it to PDFs, and making it follow a specific sequence of steps. He built LangChain to standardize these "chains" of events, and it quickly became the fastest-growing open-source project in GitHub history at the time. Powered by LLMs like GPT-4, Llama 4, and Gemini, LangChain allows developers to 'chain' together different components—such as prompt tem